# ProAID — B2.3: Threshold Sensitivity & Stability Analysis
**PT-L2Detect Benchmark** | Google Colab + GPU T4

Phân tích độ nhạy của calibration với 2 cấp độ:
1. **Single-threshold sweep**: Vary τ=0.5 ± 0.30 → F1-FPR trade-off cho từng CEFR (baseline comparison)
2. **Per-group perturbation**: Fix 2 threshold tại optimal, vary threshold còn lại ± σ → đo cross-group coupling
3. **Stability analysis**: Xác định stable region (≥95% max F1) cho mỗi group

**Prerequisite**: B1 + B2.2 models trên Google Drive (`MyDrive/proaid/models/`)

## 0. Setup & Mount Drive

In [ ]:
!pip install -q transformers scikit-learn torch matplotlib seaborn

In [ ]:
import json, os, re, numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, confusion_matrix, accuracy_score
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/proaid'
DATA_DIR = f'{DRIVE_BASE}/data'
MODEL_DIR = f'{DRIVE_BASE}/models'
LOG_DIR = f'{DRIVE_BASE}/logs'

print(f'Data:  {os.listdir(DATA_DIR)}')
print(f'Models: {os.listdir(MODEL_DIR)}')

## 1. Load All Models & Reconstruct Expected Stats from Drive

In [ ]:
CEFR_3CLASS = ["Beginner", "Intermediate", "Advanced"]
CEFR_TO_3CLASS = {"A1":"Beginner","A2":"Beginner","B1":"Intermediate","B2":"Intermediate","C1":"Advanced"}
LING_DIM = GAP_DIM = 8

# === B1 CEFR Classifier ===
class CEFRClassifier3(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden,128),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_labels))
    def forward(self, ids, am):
        return self.classifier(self.encoder(input_ids=ids, attention_mask=am).last_hidden_state[:,0,:])

cefr_ckpt = torch.load(f'{MODEL_DIR}/best_model.pt', map_location=DEVICE)
cefr_model = CEFRClassifier3(cefr_ckpt.get('model_name','xlm-roberta-base'), 3).to(DEVICE)
cefr_model.load_state_dict(cefr_ckpt['model_state_dict'])
cefr_model.eval()
cefr_tokenizer = AutoTokenizer.from_pretrained(cefr_ckpt.get('model_name','xlm-roberta-base'))

# === B2.2 Detector ===
class AIDetectorCalibrated(nn.Module):
    def __init__(self, model_name, ling_dim=8, gap_dim=8, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        total_dim = self.encoder.config.hidden_size + ling_dim + gap_dim
        self.classifier = nn.Sequential(nn.Linear(total_dim,256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,1))
    def forward(self, ids, am, lf, gf):
        xlmr_cls = self.encoder(input_ids=ids, attention_mask=am).last_hidden_state[:,0,:]
        return self.classifier(torch.cat([xlmr_cls, lf, gf], dim=-1)).squeeze(-1)

det_ckpt = torch.load(f'{MODEL_DIR}/calibrated_detector.pt', map_location=DEVICE)
detector = AIDetectorCalibrated('xlm-roberta-base', LING_DIM, GAP_DIM).to(DEVICE)
detector.load_state_dict(det_ckpt['model_state_dict'])
detector.eval()
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

print('✅ Both models loaded from Drive')

## 2. Prepare Test Data with All Features

In [ ]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def extract_linguistic_features(text: str) -> np.ndarray:
    words = text.split(); n_words = len(words)
    f_avg_word_len = np.mean([len(w) for w in words]) if words else 0
    f_ttr = len(set(w.lower() for w in words)) / n_words if n_words else 0
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    n_sentences = max(len(sentences), 1)
    f_avg_sent_len = n_words / n_sentences
    f_punct_ratio = sum(1 for c in text if c in ',;:()"\'-') / len(text) if text else 0
    f_upper_ratio = sum(1 for c in text if c.isupper()) / len(text) if text else 0
    f_comma_per_sent = text.count(',') / n_sentences
    f_short_word_ratio = sum(1 for w in words if len(w) <= 3) / n_words if n_words else 0
    return np.array([n_words, f_avg_word_len, f_ttr, f_avg_sent_len,
                     f_punct_ratio, f_upper_ratio, f_comma_per_sent, f_short_word_ratio], dtype=np.float32)

test_data = load_jsonl(f'{DATA_DIR}/test_full.jsonl')
train_data = load_jsonl(f'{DATA_DIR}/train_full.jsonl')

# Predict CEFR 3-class
print('🔮 Predicting CEFR labels...')
texts = [e['text'] for e in test_data]
pred_classes = []
for i in range(0, len(texts), 32):
    bt = texts[i:i+32]
    enc = cefr_tokenizer(bt, truncation=True, padding=True, max_length=512, return_tensors='pt')
    with torch.no_grad():
        logits = cefr_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
        pred_classes.extend([CEFR_3CLASS[p] for p in torch.argmax(logits,-1).cpu().numpy()])

# Extract features + CEFR-gap using TRAIN expected stats
for e, pc in zip(test_data, pred_classes):
    e['pred_class'] = pc
    e['ling_feats'] = extract_linguistic_features(e['text'])

# Compute expected stats from TRAIN (need to predict train CEFR too)
train_texts = [e['text'] for e in train_data]
train_preds = []
for i in range(0, len(train_texts), 32):
    bt = train_texts[i:i+32]
    enc = cefr_tokenizer(bt, truncation=True, padding=True, max_length=512, return_tensors='pt')
    with torch.no_grad():
        logits = cefr_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
        train_preds.extend([CEFR_3CLASS[p] for p in torch.argmax(logits,-1).cpu().numpy()])

for e, pc in zip(train_data, train_preds):
    e['pred_class'] = pc
    e['ling_feats'] = extract_linguistic_features(e['text'])

expected_stats = {}
for grp in CEFR_3CLASS:
    grp_feats = np.array([e['ling_feats'] for e in train_data if e['pred_class'] == grp])
    if len(grp_feats) == 0: continue
    expected_stats[grp] = {'mean': grp_feats.mean(0), 'std': grp_feats.std(0) + 1e-8}

for e in test_data:
    grp = e['pred_class']
    if grp in expected_stats:
        e['gap_feats'] = ((e['ling_feats'] - expected_stats[grp]['mean']) / expected_stats[grp]['std']).astype(np.float32)
    else:
        e['gap_feats'] = np.zeros(LING_DIM, dtype=np.float32)

print(f'✅ Prepared {len(test_data)} test essays')

In [ ]:
# Run inference
all_probs, all_labels, all_cefrs, all_classes = [], [], [], []
for i in range(0, len(test_data), 16):
    batch = test_data[i:i+16]
    bt = [e['text'] for e in batch]
    enc = tokenizer(bt, truncation=True, padding=True, max_length=512, return_tensors='pt')
    lf = torch.tensor(np.stack([e['ling_feats'] for e in batch]), dtype=torch.float)
    gf = torch.tensor(np.stack([e['gap_feats'] for e in batch]), dtype=torch.float)
    with torch.no_grad():
        logits = detector(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE), lf.to(DEVICE), gf.to(DEVICE))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
    all_labels.extend([1.0 if e['is_ai'] else 0.0 for e in batch])
    all_cefrs.extend([e['cefr_level'] for e in batch])
    all_classes.extend([e['pred_class'] for e in batch])

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
print(f'✅ Inference done: {len(all_probs)} essays')

## 3. Threshold Sensitivity: F1-FPR Trade-off

In [ ]:
sigmas = np.arange(-0.30, 0.31, 0.02)
center_tau = 0.5
results = []

for sigma in sigmas:
    tau = center_tau + sigma
    preds = (all_probs >= tau).astype(int)
    per_cefr = {}
    for lvl in ['A1','A2','B1','B2','C1']:
        mask = np.array([c == lvl for c in all_cefrs])
        if mask.sum() == 0: continue
        g_l, g_p = all_labels[mask], preds[mask]
        tn, fp, fn, tp = confusion_matrix(g_l, g_p, labels=[0,1]).ravel()
        per_cefr[lvl] = {'f1': f1_score(g_l,g_p), 'fpr': fp/(fp+tn) if (fp+tn)>0 else 0}
    results.append({'tau': tau, 'sigma': sigma,
        'global_f1': f1_score(all_labels, preds),
        'global_fpr': sum((preds==1)&(all_labels==0))/max(sum(all_labels==0),1),
        'per_cefr': per_cefr})

print('🔍 Optimal τ per CEFR level from sensitivity sweep:')
optimal_taus_dict = {}
for lvl in ['A1','A2','B1','B2','C1']:
    f1s = np.array([r['per_cefr'][lvl]['f1'] for r in results if lvl in r['per_cefr']])
    taus = np.array([r['tau'] for r in results if lvl in r['per_cefr']])
    best_t, best_f1 = taus[f1s.argmax()], f1s.max()
    optimal_taus_dict[lvl] = best_t
    print(f'  {lvl}: τ_opt = {best_t:.3f} (F1 = {best_f1:.4f})')

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'A1':'red','A2':'orange','B1':'green','B2':'blue','C1':'purple'}

for lvl in ['A1','A2','B1','B2','C1']:
    taus = [r['tau'] for r in results if lvl in r['per_cefr']]
    f1s = [r['per_cefr'][lvl]['f1'] for r in results if lvl in r['per_cefr']]
    axes[0].plot(taus, f1s, color=colors[lvl], marker='.', label=lvl)
    best_idx = np.argmax(f1s)
    axes[0].scatter([taus[best_idx]], [f1s[best_idx]], color=colors[lvl], s=80, zorder=5, edgecolors='black')
axes[0].set_xlabel('Threshold τ'); axes[0].set_ylabel('F1')
axes[0].set_title('F1 vs Threshold per CEFR', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for lvl in ['A1','A2','B1','B2','C1']:
    taus = [r['tau'] for r in results if lvl in r['per_cefr']]
    fprs = [r['per_cefr'][lvl]['fpr'] for r in results if lvl in r['per_cefr']]
    axes[1].plot(taus, fprs, color=colors[lvl], marker='.', label=lvl)
axes[1].set_xlabel('Threshold τ'); axes[1].set_ylabel('FPR')
axes[1].set_title('FPR vs Threshold per CEFR', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

global_f1s = [r['global_f1'] for r in results]
global_fprs = [r['global_fpr'] for r in results]
scatter = axes[2].scatter(global_fprs, global_f1s, c=[r['tau'] for r in results], cmap='viridis', s=30)
axes[2].set_xlabel('Global FPR'); axes[2].set_ylabel('Global F1')
axes[2].set_title('F1-FPR Trade-off (Global)', fontweight='bold')
plt.colorbar(scatter, ax=axes[2], label='τ')
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f'{LOG_DIR}/sensitivity_analysis.png', dpi=150); plt.show()
print(f'💾 Saved: {LOG_DIR}/sensitivity_analysis.png')

## 5. Stability Analysis

In [ ]:
print('📊 Threshold Stability (F1 within 95% of max):\n')
for lvl in ['A1','A2','B1','B2','C1']:
    f1s = np.array([r['per_cefr'][lvl]['f1'] for r in results if lvl in r['per_cefr']])
    taus = np.array([r['tau'] for r in results if lvl in r['per_cefr']])
    best_f1 = f1s.max()
    close_mask = f1s >= 0.95 * best_f1
    stable_taus = taus[close_mask]
    print(f'  {lvl}: best τ={taus[f1s.argmax()]:.3f} (F1={best_f1:.4f})')
    print(f'         95%-stable: [{stable_taus.min():.3f}, {stable_taus.max():.3f}] (width={stable_taus.max()-stable_taus.min():.3f})')

print(f'\n📋 Recommended τ per 3-class group:')
grp_map = {'Beginner': ['A1','A2'], 'Intermediate': ['B1','B2'], 'Advanced': ['C1']}
for grp, lvls in grp_map.items():
    best_taus = []
    for l in lvls:
        f1s = np.array([r['per_cefr'][l]['f1'] for r in results if l in r['per_cefr']])
        taus = np.array([r['tau'] for r in results if l in r['per_cefr']])
        best_taus.append(taus[f1s.argmax()])
    print(f'  {grp}: avg τ = {np.mean(best_taus):.3f} (from {lvls})')

## 6. Save Results to Drive

In [ ]:
log = {
    'experiment': 'B2.3 — Threshold Sensitivity Analysis',
    'timestamp': datetime.now().isoformat(),
    'center_tau': center_tau,
    'sigma_range': [-0.30, 0.30],
    'num_points': len(sigmas),
    'optimal_taus_per_cefr': {l: float(t) for l, t in optimal_taus_dict.items()},
    'recommended_3class_taus': {}
}
for grp, lvls in grp_map.items():
    log['recommended_3class_taus'][grp] = float(np.mean([optimal_taus_dict[l] for l in lvls]))

with open(f'{LOG_DIR}/experiment_log_b2.3.json', 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

print(f'✅ All saved to Drive:')
print(f'   Log:  {LOG_DIR}/experiment_log_b2.3.json')
print(f'   Plot: {LOG_DIR}/sensitivity_analysis.png')